# Pandas Index & Axis — Clear Guide

This notebook explains what the `Index` is in pandas, how `axis` is used across pandas APIs (rows vs columns), contrasts with NumPy axes, and shows practical examples you’ll use in data analysis and interviews.

## Quick setup

In [1]:
import pandas as pd
import numpy as np
pd.__version__

'3.0.0'

## Axis contrast with NumPy

- NumPy `axis` has same numeric meaning but beware: for 2D arrays axis=0 is down (rows), axis=1 is across (columns). pandas follows this convention.
- Example: `np.sum(arr, axis=1)` sums across columns for each row, same as `df.sum(axis=1)` for a DataFrame with similar layout.

In [2]:
arr = np.array([[1,2,3],[4,5,6]])
print('np sum axis=0 ->', arr.sum(axis=0))
print('np sum axis=1 ->', arr.sum(axis=1))
print('as DataFrame:')
df_arr = pd.DataFrame(arr, columns=['x','y','z'])
print(df_arr.sum(axis=0).values, df_arr.sum(axis=1).values)


np sum axis=0 -> [5 7 9]
np sum axis=1 -> [ 6 15]
as DataFrame:
[5 7 9] [ 6 15]


## Axis in concatenation and merging

- `pd.concat([df1, df2], axis=0)` appends rows (stack vertically); `axis=1` concatenates columns (side-by-side).
- `merge` primarily uses keys/columns (not `axis`), but `join` can join on index (so index alignment matters).

In [3]:
left = pd.DataFrame({'id':[1,2], 'v':[10,20]}).set_index('id')
right = pd.DataFrame({'id':[2,3], 'w':[200,300]}).set_index('id')
print('join on index ->', left.join(right, how='outer'))
print('concat axis=0 ->', pd.concat([left.reset_index(), right.reset_index()], axis=0))
print('concat axis=1 ->', pd.concat([left, right], axis=1))


join on index ->        v      w
id             
1   10.0    NaN
2   20.0  200.0
3    NaN  300.0
concat axis=0 ->    id     v      w
0   1  10.0    NaN
1   2  20.0    NaN
0   2   NaN  200.0
1   3   NaN  300.0
concat axis=1 ->        v      w
id             
1   10.0    NaN
2   20.0  200.0
3    NaN  300.0


## MultiIndex and axis levels

- For `MultiIndex`, axis still refers to rows vs columns, but you can address levels within an index. `df.swaplevel(i,j, axis=0)` swaps index levels for rows; you can also have MultiIndex on columns (axis=1).
- Many operations accept `level=` to apply across a specific index level.

In [4]:
mi_df = pd.DataFrame({'city':['ny','ny','la','la'], 'date':['2020-01-01','2020-01-02','2020-01-01','2020-01-02'], 'v':[1,2,3,4]})
mi_df = mi_df.set_index(['city','date'])
print(mi_df)
print('unstack (move inner index to columns) ->', mi_df.unstack(level=1))
print('swaplevel (no-op visible when sorting):', mi_df.swaplevel(0,1))


                 v
city date         
ny   2020-01-01  1
     2020-01-02  2
la   2020-01-01  3
     2020-01-02  4
unstack (move inner index to columns) ->               v           
date 2020-01-01 2020-01-02
city                      
la            3          4
ny            1          2
swaplevel (no-op visible when sorting):                  v
date       city   
2020-01-01 ny    1
2020-01-02 ny    2
2020-01-01 la    3
2020-01-02 la    4


## Common gotchas & best practices

- Remember `loc` uses labels (index/column names), `iloc` uses integer positions. Mixing them causes confusion.
- When using `axis`, prefer explicit integers (`axis=0` or `axis=1`) or keywords where supported (some functions accept `index`/`columns`).
- Use `df.columns` and `df.index` to inspect axes; `df.shape` gives (n_rows, n_columns).
- Sorting the index (`sort_index()`) enables fast range slicing on a monotonic index (useful for time-series).

## Practical examples — short exercises (try before viewing solutions):

1. Use `axis` to compute row-wise z-scores for a numeric DataFrame.
2. Concatenate two DataFrames side-by-side and then drop a column using `axis=1`.
3. Create a `DatetimeIndex` and resample weekly (show how index matters).